# Feature Engineering with Transformers

## Why Feature Engineering Matters

In quantitative finance, the saying goes: **"Garbage in, garbage out."** But it's more nuanced than that. Even with good data, raw features are rarely optimal for modeling.

Consider these challenges:
- **Scale differences**: Market cap ($billions) vs P/E ratio (single digits)
- **Missing data**: Not all stocks have all fundamentals reported
- **Categorical data**: Sector, country, exchange - can't feed strings to a model
- **Time series**: Today's return might depend on yesterday's, last week's, or last month's

**Transformers** solve these problems. In this notebook, you'll learn:
- Which scaler to use (and when scaling matters)
- How to encode categorical variables properly
- Strategies for handling missing data
- **Time-series feature engineering** - lags, rolling stats, momentum signals
- How to build your own custom transformers
- How to apply different transformations to different columns

Let's dive in.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, 
    OneHotEncoder, OrdinalEncoder, 
    QuantileTransformer, PowerTransformer
)

# Imputation
from sklearn.impute import SimpleImputer, KNNImputer

# Custom transformers
from sklearn.base import BaseEstimator, TransformerMixin

# Column transformers and pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline

# For evaluation
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Reproducibility
np.random.seed(42)

# Plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Ready for feature engineering!")

## Generate Sample Data

We'll create a realistic financial dataset with different types of features.

In [ ]:
def generate_stock_features(n_stocks=500, seed=42):
    """Generate a cross-sectional dataset of stock features."""
    np.random.seed(seed)
    
    # Numeric features (varying scales and distributions)
    df = pd.DataFrame({
        # Market cap: heavily right-skewed (most are small, few mega-caps)
        'market_cap': np.random.lognormal(8, 2, n_stocks),
        
        # P/E ratio: can be negative (losses), high (growth), or normal
        'pe_ratio': np.random.normal(20, 30, n_stocks),
        
        # Book-to-market: right-skewed, always positive
        'book_to_market': np.abs(np.random.normal(0.5, 0.3, n_stocks)),
        
        # Momentum (past 12m return): roughly normal
        'momentum_12m': np.random.normal(0.12, 0.35, n_stocks),
        
        # Volatility: always positive, right-skewed
        'volatility': np.abs(np.random.normal(0.30, 0.15, n_stocks)),
        
        # Turnover ratio
        'turnover': np.random.exponential(0.05, n_stocks),
    })
    
    # Categorical features
    sectors = ['Technology', 'Finance', 'Healthcare', 'Consumer', 'Energy', 'Industrial']
    df['sector'] = np.random.choice(sectors, n_stocks)
    
    exchanges = ['NYSE', 'NASDAQ', 'AMEX']
    df['exchange'] = np.random.choice(exchanges, n_stocks, p=[0.4, 0.5, 0.1])
    
    # Add some missing values (realistic!)
    df.loc[np.random.choice(n_stocks, 50, replace=False), 'pe_ratio'] = np.nan
    df.loc[np.random.choice(n_stocks, 30, replace=False), 'book_to_market'] = np.nan
    df.loc[np.random.choice(n_stocks, 20, replace=False), 'turnover'] = np.nan
    
    # Target: next month return (depends on features + noise)
    df['next_month_return'] = (
        -0.05 * np.log(df['market_cap']) / np.log(df['market_cap']).std() +
        0.02 * df['book_to_market'] / df['book_to_market'].std() +
        0.01 * df['momentum_12m'] / df['momentum_12m'].std() -
        0.03 * df['volatility'] / df['volatility'].std() +
        np.random.normal(0, 0.05, n_stocks)
    )
    
    return df

df = generate_stock_features()
print(f"Generated {len(df)} stocks")
print(f"Missing values:\n{df.isnull().sum()}")
df.head(10)

---

## Part 1: Scaling and Normalization

### Why Scale Features?

Many ML algorithms are **sensitive to feature scale**:
- **Distance-based methods** (KNN, SVM): Large-scale features dominate
- **Regularized models** (Ridge, Lasso): Penalize coefficients equally only if features are on same scale
- **Gradient descent**: Converges faster with normalized features

Some algorithms **don't care about scale**:
- Tree-based methods (Random Forest, XGBoost)
- Plain linear regression (OLS) without regularization

**Rule of thumb**: When in doubt, scale. It rarely hurts and often helps.

In [ ]:
# Look at the raw feature distributions
numeric_cols = ['market_cap', 'pe_ratio', 'book_to_market', 'momentum_12m', 'volatility', 'turnover']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    df[col].hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col}\nMean: {df[col].mean():.2f}, Std: {df[col].std():.2f}')
    ax.set_xlabel('')
plt.suptitle('Raw Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()

### 1.1 StandardScaler: The Default Choice

Transforms features to have **mean = 0** and **standard deviation = 1**.

$$z = \frac{x - \mu}{\sigma}$$

**Use when**: Features are roughly normally distributed, no extreme outliers.

In [ ]:
# Create sample data without NaN for scaling demos
X_sample = df[numeric_cols].dropna()

# StandardScaler
scaler_standard = StandardScaler()
X_standard = scaler_standard.fit_transform(X_sample)

print("StandardScaler:")
print(f"  Before - Mean: {X_sample.mean().round(2).tolist()}")
print(f"  Before - Std:  {X_sample.std().round(2).tolist()}")
print(f"  After  - Mean: {X_standard.mean(axis=0).round(4).tolist()}")
print(f"  After  - Std:  {X_standard.std(axis=0).round(4).tolist()}")

### 1.2 MinMaxScaler: Bounded Features

Scales features to a fixed range, typically [0, 1].

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

**Use when**: You need bounded outputs, or your model expects values in [0, 1] (like neural network inputs).

In [ ]:
# MinMaxScaler
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
X_minmax = scaler_minmax.fit_transform(X_sample)

print("MinMaxScaler:")
print(f"  Min: {X_minmax.min(axis=0).round(4).tolist()}")
print(f"  Max: {X_minmax.max(axis=0).round(4).tolist()}")

### 1.3 RobustScaler: When You Have Outliers

Uses **median** and **interquartile range (IQR)** instead of mean and std.

$$x_{scaled} = \frac{x - \text{median}}{\text{IQR}}$$

**Use when**: You have significant outliers that you don't want to remove. Very common in finance!

In [ ]:
# RobustScaler
scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(X_sample)

print("RobustScaler:")
print(f"  Median: {np.median(X_robust, axis=0).round(4).tolist()}")
print(f"  IQR:    {(np.percentile(X_robust, 75, axis=0) - np.percentile(X_robust, 25, axis=0)).round(4).tolist()}")

In [ ]:
# Compare how different scalers handle market_cap
# Key insight: LinearScalers preserve shape, but change scale!

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: Histograms
data_list = [
    (X_sample['market_cap'].values, 'Original', 'steelblue'),
    (X_standard[:, 0], 'StandardScaler', 'orange'),
    (X_minmax[:, 0], 'MinMaxScaler', 'green'),
    (X_robust[:, 0], 'RobustScaler', 'red'),
]

for ax, (data, title, color) in zip(axes[0], data_list):
    ax.hist(data, bins=50, edgecolor='black', alpha=0.7, color=color)
    ax.axvline(x=np.mean(data), color='black', linestyle='-', linewidth=2, label=f'Mean: {np.mean(data):.2f}')
    ax.axvline(x=np.median(data), color='darkgray', linestyle='--', linewidth=2, label=f'Median: {np.median(data):.2f}')
    ax.set_title(f'{title}\nRange: [{data.min():.1f}, {data.max():.1f}]')
    ax.legend(fontsize=8)

# Bottom row: Box plots (better for comparing scale!)
axes[1][0].boxplot(X_sample['market_cap'].values, vert=True)
axes[1][0].set_title(f'Original\nStd: {X_sample["market_cap"].std():.0f}')
axes[1][0].set_ylabel('Value')

axes[1][1].boxplot(X_standard[:, 0], vert=True)
axes[1][1].set_title(f'StandardScaler\nStd: {X_standard[:, 0].std():.2f}')

axes[1][2].boxplot(X_minmax[:, 0], vert=True)
axes[1][2].set_title(f'MinMaxScaler\nRange: [0, 1]')

axes[1][3].boxplot(X_robust[:, 0], vert=True)
axes[1][3].set_title(f'RobustScaler\nIQR: {np.percentile(X_robust[:, 0], 75) - np.percentile(X_robust[:, 0], 25):.2f}')

plt.suptitle('Scaling Comparison: market_cap\\n(Notice: Shape stays the same, only SCALE changes!)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

print("Key insight: Linear scalers (Standard, MinMax, Robust) preserve distribution SHAPE.")
print("They only change the SCALE (mean, range, spread).")
print("To change the shape, you need QuantileTransformer (shown below).")

**Observation**: 
- StandardScaler and MinMaxScaler are heavily influenced by the extreme outliers
- RobustScaler is more stable, spreading the bulk of the data better

### 1.4 QuantileTransformer: Forcing a Distribution

Maps your data to a specified distribution (uniform or normal). Useful when you have very non-normal data.

In [ ]:
# QuantileTransformer to uniform distribution
qt_uniform = QuantileTransformer(n_quantiles=min(1000, len(X_sample)), output_distribution='uniform', random_state=42)
X_uniform = qt_uniform.fit_transform(X_sample)

# QuantileTransformer to normal distribution
qt_normal = QuantileTransformer(n_quantiles=min(1000, len(X_sample)), output_distribution='normal', random_state=42)
X_qt_normal = qt_normal.fit_transform(X_sample)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(X_sample['market_cap'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Original market_cap')

axes[1].hist(X_uniform[:, 0], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_title('QuantileTransformer (uniform)')

axes[2].hist(X_qt_normal[:, 0], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[2].set_title('QuantileTransformer (normal)')

plt.tight_layout()

### Scaler Cheat Sheet

| Scaler | When to Use | Pros | Cons |
|--------|-------------|------|------|
| **StandardScaler** | Default choice, roughly normal data | Fast, interpretable | Sensitive to outliers |
| **MinMaxScaler** | Need bounded [0,1] output | Preserves shape | Very sensitive to outliers |
| **RobustScaler** | Data has outliers | Robust to extremes | May not center at 0 |
| **QuantileTransformer** | Very skewed data | Forces desired distribution | Nonlinear, loses interpretability |

---

## Part 2: Encoding Categorical Variables

Models need numbers, not strings. How do we handle categorical features like "sector" or "exchange"?

In [ ]:
# Look at our categorical features
print("Sectors:")
print(df['sector'].value_counts())
print("\nExchanges:")
print(df['exchange'].value_counts())

### 2.1 OrdinalEncoder: When Order Matters

Maps categories to integers: 0, 1, 2, ...

**Use when**: Categories have a natural order (e.g., credit rating: AAA > AA > A > BBB)

In [ ]:
# Example: Credit ratings have natural order
ratings = np.array([['AA'], ['BBB'], ['A'], ['AAA'], ['BB'], ['A']])

# Specify the order explicitly
encoder_ordinal = OrdinalEncoder(categories=[['BB', 'BBB', 'A', 'AA', 'AAA']])
ratings_encoded = encoder_ordinal.fit_transform(ratings)

print("Credit Rating Encoding:")
for original, encoded in zip(ratings.flatten(), ratings_encoded.flatten()):
    print(f"  {original} -> {int(encoded)}")

### 2.2 OneHotEncoder: When Order Doesn't Matter

Creates a binary column for each category.

**Use when**: Categories have no natural order (e.g., sector, country)

In [ ]:
# One-hot encode sectors
encoder_onehot = OneHotEncoder(sparse_output=False)  # sparse_output=False for dense array
sectors_encoded = encoder_onehot.fit_transform(df[['sector']])

print(f"Original shape: {df[['sector']].shape}")
print(f"Encoded shape: {sectors_encoded.shape}")
print(f"\nCategories: {encoder_onehot.categories_[0]}")
print(f"\nFirst 5 rows:")
print(pd.DataFrame(sectors_encoded, columns=encoder_onehot.get_feature_names_out()).head())

### 2.3 Drop One Category (Avoiding Multicollinearity)

**The Problem**: With one-hot encoding, if you know 5 of the 6 sector columns, you can perfectly predict the 6th (they must sum to 1). This is called **perfect multicollinearity**.

**Why is multicollinearity bad?**

1. **Math breaks**: Linear regression computes $(X^T X)^{-1}$. If columns are perfectly correlated, this matrix is **singular** (non-invertible). The regression literally cannot run, or gives garbage results.

2. **Coefficients become meaningless**: Even with near-perfect (not exact) multicollinearity, small changes in data cause huge swings in coefficients. You might get Technology = +0.5 and Finance = -10.3 one day, then Technology = -8.7 and Finance = +2.1 the next.

3. **You can't interpret effects**: If Technology goes up by 1, what happens to the prediction? It depends on what else changed — because the columns aren't independent.

**The Solution**: Drop one category. It becomes the **reference category**, and all other coefficients are interpreted *relative to it*.

```
If we drop "Consumer":
- Technology coefficient = 0.02 means "Tech stocks return 2% more than Consumer stocks"  
- Finance coefficient = -0.01 means "Finance stocks return 1% less than Consumer stocks"
```

This is mathematically equivalent but now the matrix is invertible and coefficients are interpretable.

In [ ]:
# Drop first category (it becomes the "reference")
encoder_onehot_drop = OneHotEncoder(sparse_output=False, drop='first')
sectors_encoded_drop = encoder_onehot_drop.fit_transform(df[['sector']])

print(f"Shape with drop='first': {sectors_encoded_drop.shape}")
print(f"Feature names: {encoder_onehot_drop.get_feature_names_out().tolist()}")
print("\nInterpretation: All coefficients are relative to the dropped category (Consumer)")

---

## Part 3: Handling Missing Data

Real financial data is messy. Not all companies report all metrics. How do we handle this?

In [ ]:
# Check missing values
print("Missing values per column:")
print(df[numeric_cols].isnull().sum())
print(f"\nTotal rows: {len(df)}")

### 3.1 SimpleImputer: Basic Strategies

Replace missing values with a statistic computed from the non-missing values.

In [ ]:
# Mean imputation
imputer_mean = SimpleImputer(strategy='mean')
X_imputed_mean = imputer_mean.fit_transform(df[numeric_cols])

# Median imputation (more robust to outliers)
imputer_median = SimpleImputer(strategy='median')
X_imputed_median = imputer_median.fit_transform(df[numeric_cols])

# Check: no more missing values
print(f"Missing values after mean imputation: {np.isnan(X_imputed_mean).sum()}")
print(f"Missing values after median imputation: {np.isnan(X_imputed_median).sum()}")

In [ ]:
# See the imputed values for pe_ratio
pe_idx = numeric_cols.index('pe_ratio')
print(f"PE Ratio imputation values:")
print(f"  Mean: {imputer_mean.statistics_[pe_idx]:.2f}")
print(f"  Median: {imputer_median.statistics_[pe_idx]:.2f}")
print(f"\nMedian is better here because PE ratio has outliers (negative and very high values)")

### 3.2 KNNImputer: Using Similar Observations

Impute missing values using the mean of the K nearest neighbors. This uses information from related stocks!

**How KNNImputer Works:**

1. **Find neighbors**: For a row with missing values, compute distance to all other rows using only the features that are *not* missing in both rows.

2. **Distance metric**: By default, uses **Euclidean distance**:
   $$d(x, y) = \sqrt{\sum_{i \in \text{valid}} (x_i - y_i)^2}$$
   where "valid" means features that are non-missing in both rows.

3. **Select K nearest**: Pick the K rows with smallest distance.

4. **Impute**: Fill the missing value with the **mean** of that feature across the K neighbors.

**Example**: Stock A is missing PE ratio. KNNImputer finds 5 stocks with similar market cap, momentum, and volatility. Their PE ratios are [15, 18, 12, 20, 16]. Stock A gets PE = mean([15, 18, 12, 20, 16]) = 16.2.

**Why this is smart for finance**: A small-cap tech stock with high momentum probably has a similar PE ratio to other small-cap tech stocks with high momentum — not to the average of all stocks!

In [ ]:
# KNN Imputation
imputer_knn = KNNImputer(n_neighbors=5)
X_imputed_knn = imputer_knn.fit_transform(df[numeric_cols])

# Compare imputation results for a specific stock with missing PE ratio
missing_pe_idx = df[df['pe_ratio'].isnull()].index[0]

print(f"Stock {missing_pe_idx} has missing PE ratio")
print(f"\nImputed values:")
print(f"  Mean imputation: {X_imputed_mean[missing_pe_idx, pe_idx]:.2f}")
print(f"  KNN imputation:  {X_imputed_knn[missing_pe_idx, pe_idx]:.2f}")
print(f"\nKNN uses other features (market cap, momentum, etc.) to find similar stocks!")

### 3.3 Adding Missing Indicators

Sometimes, the fact that data is missing is informative! (e.g., small companies might not report certain metrics)

In [ ]:
# Create missing indicators
from sklearn.impute import SimpleImputer

imputer_with_indicator = SimpleImputer(strategy='median', add_indicator=True)
X_with_indicators = imputer_with_indicator.fit_transform(df[numeric_cols])

print(f"Original features: {len(numeric_cols)}")
print(f"With indicators: {X_with_indicators.shape[1]}")
print(f"\nThe extra columns are binary indicators for which values were missing.")

---

## Part 4: Time-Series Feature Engineering

**This is crucial for finance.** Financial data is inherently sequential. Features often depend on past values.

Common time-series features:
- **Lag features**: What was yesterday's return?
- **Rolling statistics**: What's the average return over the last 20 days?
- **Momentum/change**: How much did the price change over the last month?

In [ ]:
def generate_time_series_data(n_days=500, seed=42):
    """Generate a time series of stock returns."""
    np.random.seed(seed)
    
    dates = pd.date_range(end='2024-12-31', periods=n_days, freq='B')
    
    # Generate price series with some autocorrelation in volatility
    returns = np.random.normal(0.0004, 0.02, n_days)  # ~10% annual return, 20% vol
    
    # Add some volatility clustering (GARCH-like effect)
    vol_factor = np.ones(n_days)
    for i in range(1, n_days):
        vol_factor[i] = 0.9 * vol_factor[i-1] + 0.1 * (abs(returns[i-1]) / 0.02)
    returns = returns * vol_factor
    
    prices = 100 * np.exp(np.cumsum(returns))
    
    df = pd.DataFrame({
        'date': dates,
        'price': prices,
        'return': returns,
        'volume': np.random.lognormal(15, 0.5, n_days),
    })
    df.set_index('date', inplace=True)
    
    return df

ts_data = generate_time_series_data()
print(f"Date range: {ts_data.index[0]} to {ts_data.index[-1]}")
ts_data.head(10)

### 4.1 Lag Features: What Happened Before?

Lag features capture serial dependencies. "What was the return yesterday?" "What was it 5 days ago?"

In [ ]:
# Create lag features
ts_features = ts_data.copy()

# Lag 1 day (yesterday's return)
ts_features['return_lag1'] = ts_features['return'].shift(1)

# Lag 5 days (last week's return)
ts_features['return_lag5'] = ts_features['return'].shift(5)

# Lag 20 days (last month's return)
ts_features['return_lag20'] = ts_features['return'].shift(20)

ts_features['future'] = ts_features['return'].shift(-1)

print("Lag features (note NaN in early rows):")
ts_features[['return', 'return_lag1', 'return_lag5', 'return_lag20', 'future']].head(25)

### Critical Warning: Lookahead Bias!

**NEVER use `shift(-1)` or negative lags** for features. That's future data!

```python
# WRONG - This is future return!
df['future_return'] = df['return'].shift(-1)  # DON'T DO THIS!

# RIGHT - This is past return
df['past_return'] = df['return'].shift(1)  # CORRECT
```

### 4.2 Rolling Statistics: Smoothed Signals

Rolling windows compute statistics over a trailing window of observations.

In [ ]:
# Rolling mean (moving average)
ts_features['return_ma5'] = ts_features['return'].rolling(window=5).mean()
ts_features['return_ma20'] = ts_features['return'].rolling(window=20).mean()

# Rolling volatility (standard deviation)
ts_features['volatility_5d'] = ts_features['return'].rolling(window=5).std()
ts_features['volatility_20d'] = ts_features['return'].rolling(window=20).std()

# Rolling max/min (for drawdown calculations)
ts_features['price_max_20d'] = ts_features['price'].rolling(window=20).max()

print("Rolling features:")
ts_features[['return', 'return_ma5', 'return_ma20', 'volatility_20d']].tail(10)

In [ ]:
# Visualize rolling features
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price with moving averages
axes[0].plot(ts_features.index, ts_features['price'], label='Price', alpha=0.7)
price_ma20 = ts_features['price'].rolling(20).mean()
price_ma50 = ts_features['price'].rolling(50).mean()
axes[0].plot(ts_features.index, price_ma20, label='20-day MA', linewidth=2)
axes[0].plot(ts_features.index, price_ma50, label='50-day MA', linewidth=2)
axes[0].set_ylabel('Price')
axes[0].set_title('Price with Moving Averages')
axes[0].legend()

# Rolling volatility
axes[1].fill_between(ts_features.index, 0, ts_features['volatility_20d'] * np.sqrt(252) * 100, 
                     alpha=0.5, color='red', label='20-day Annualized Vol')
axes[1].set_ylabel('Volatility (%)')
axes[1].set_title('Rolling Volatility')
axes[1].legend()

plt.tight_layout()

### 4.3 Momentum and Rate of Change

How much has the price changed over a period? This is the classic "momentum" factor.

In [ ]:
# Momentum: percentage change over N periods
ts_features['momentum_5d'] = ts_features['price'].pct_change(5)
ts_features['momentum_20d'] = ts_features['price'].pct_change(20)
ts_features['momentum_60d'] = ts_features['price'].pct_change(60)

# Log returns over N periods (same as cumulative return)
ts_features['log_return_20d'] = np.log(ts_features['price'] / ts_features['price'].shift(20))

print("Momentum features:")
ts_features[['price', 'momentum_5d', 'momentum_20d', 'momentum_60d']].tail(10)

### 4.4 Technical Indicators

Let's create some common technical indicators used in trading strategies.

In [ ]:
# Relative Strength Index (RSI) - measures momentum
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

ts_features['rsi_14'] = compute_rsi(ts_features['price'], 14)

# Bollinger Bands position (where is price relative to bands?)
ma20 = ts_features['price'].rolling(20).mean()
std20 = ts_features['price'].rolling(20).std()
ts_features['bollinger_position'] = (ts_features['price'] - ma20) / (2 * std20)

# MACD (Moving Average Convergence Divergence)
ema12 = ts_features['price'].ewm(span=12, adjust=False).mean()
ema26 = ts_features['price'].ewm(span=26, adjust=False).mean()
ts_features['macd'] = ema12 - ema26
ts_features['macd_signal'] = ts_features['macd'].ewm(span=9, adjust=False).mean()

print("Technical indicators:")
ts_features[['price', 'rsi_14', 'bollinger_position', 'macd']].tail(10)

In [ ]:
# Visualize RSI
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, 
                         gridspec_kw={'height_ratios': [2, 1]})

# Price
axes[0].plot(ts_features.index, ts_features['price'], label='Price')
axes[0].set_ylabel('Price')
axes[0].set_title('Price and RSI')
axes[0].legend()

# RSI
axes[1].plot(ts_features.index, ts_features['rsi_14'], label='RSI(14)', color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[1].fill_between(ts_features.index, 30, 70, alpha=0.1, color='gray')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend(loc='upper left')

plt.tight_layout()

---

## Part 5: Custom Transformers

Sometimes you need transformations that don't exist in sklearn. You can build your own!

A custom transformer needs:
1. Inherit from `BaseEstimator` and `TransformerMixin`
2. Implement `fit(self, X, y=None)` - returns `self`
3. Implement `transform(self, X)` - returns transformed data

In [ ]:
class LagFeatureTransformer(BaseEstimator, TransformerMixin):
    """Create lag features for time series data."""
    
    def __init__(self, lags=[1, 5, 20]):
        self.lags = lags
    
    def fit(self, X, y=None):
        # Nothing to learn, but must return self
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X).copy()  # Ensure DataFrame
        result = X.copy()
        
        for col in X.columns:
            for lag in self.lags:
                result[f'{col}_lag{lag}'] = X[col].shift(lag)
        
        return result
    
    def get_feature_names_out(self, input_features=None):
        """Return feature names for output."""
        if input_features is None:
            input_features = [f'x{i}' for i in range(self.n_features_in_)]
        
        output_features = list(input_features)
        for col in input_features:
            for lag in self.lags:
                output_features.append(f'{col}_lag{lag}')
        return output_features

In [ ]:
# Test our custom transformer
lag_transformer = LagFeatureTransformer(lags=[1, 5])
test_data = ts_features[['return']].head(10)

print("Original:")
print(test_data)
print("\nWith lag features:")
print(lag_transformer.fit_transform(test_data))

In [ ]:
class RollingFeatureTransformer(BaseEstimator, TransformerMixin):
    """Create rolling window features."""
    
    def __init__(self, windows=[5, 20], functions=['mean', 'std']):
        self.windows = windows
        self.functions = functions
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        result = X.copy()
        
        for col in X.columns:
            for window in self.windows:
                rolling = X[col].rolling(window=window)
                for func in self.functions:
                    result[f'{col}_rolling{window}_{func}'] = getattr(rolling, func)()
        
        return result



In [ ]:
# Test rolling transformer
rolling_transformer = RollingFeatureTransformer(windows=[3, 5], functions=['mean', 'std'])
test_data = ts_features[['return']].head(10)

print("Original:")
print(test_data.round(4))
print("\nWith rolling features:")
print(rolling_transformer.fit_transform(test_data).round(4))

### 5.1 Winsorization: Handling Extreme Outliers

A common technique in quant finance: cap extreme values at a percentile threshold.

In [ ]:
class WinsorizerTransformer(BaseEstimator, TransformerMixin):
    """Winsorize features to cap extreme values."""
    
    def __init__(self, lower_percentile=1, upper_percentile=99):
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile
    
    def fit(self, X, y=None):
        X = np.array(X)
        self.lower_bounds_ = np.percentile(X, self.lower_percentile, axis=0)
        self.upper_bounds_ = np.percentile(X, self.upper_percentile, axis=0)
        self.n_features_in_ = X.shape[1]  # Store number of features
        return self
    
    def transform(self, X):
        X = np.array(X).copy()
        for i in range(X.shape[1]):
            X[:, i] = np.clip(X[:, i], self.lower_bounds_[i], self.upper_bounds_[i])
        return X
    
    def get_feature_names_out(self, input_features=None):
        """Return feature names (unchanged since winsorization doesn't alter feature count)."""
        if input_features is None:
            # Default names if none provided
            return np.array([f'x{i}' for i in range(self.n_features_in_)])
        return np.array(input_features)

In [ ]:
# Test winsorization on market_cap
winsorizer = WinsorizerTransformer(lower_percentile=5, upper_percentile=95)
mktcap = df[['market_cap']].dropna()

mktcap_winsorized = winsorizer.fit_transform(mktcap)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(mktcap, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Original Market Cap')

axes[1].hist(mktcap_winsorized, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(x=winsorizer.lower_bounds_[0], color='r', linestyle='--', label='Lower bound')
axes[1].axvline(x=winsorizer.upper_bounds_[0], color='r', linestyle='--', label='Upper bound')
axes[1].set_title('Winsorized Market Cap (5th-95th percentile)')
axes[1].legend()

plt.tight_layout()

---

## Part 6: ColumnTransformer - Different Transforms for Different Columns

Real datasets have mixed types. `ColumnTransformer` lets you apply different transformations to different columns.

In [ ]:
# Prepare data
df_clean = df.dropna().copy()
X = df_clean.drop('next_month_return', axis=1)
y = df_clean['next_month_return']

print(f"Features: {X.columns.tolist()}")
print(f"\nNumeric: {numeric_cols}")
print(f"Categorical: ['sector', 'exchange']")

In [ ]:
# Define different preprocessing for different column types
numeric_features = ['market_cap', 'pe_ratio', 'book_to_market', 'momentum_12m', 'volatility', 'turnover']
categorical_features = ['sector', 'exchange']

# Numeric pipeline: winsorize, then scale
numeric_pipeline = Pipeline([
    ('winsorize', WinsorizerTransformer(lower_percentile=1, upper_percentile=99)),
    ('scaler', RobustScaler()),
])

# Categorical pipeline: one-hot encode
categorical_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ],
    remainder='drop'  # Drop columns not specified
)

print("Preprocessor:")
print(preprocessor)

In [ ]:
# Fit and transform
X_transformed = preprocessor.fit_transform(X)

print(f"Original shape: {X.shape}")
print(f"Transformed shape: {X_transformed.shape}")
print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features after one-hot: {X_transformed.shape[1] - len(numeric_features)}")

In [ ]:
# Get feature names
# Note: This works because our custom WinsorizerTransformer implements get_feature_names_out()
feature_names = preprocessor.get_feature_names_out()
print("Feature names after transformation:")
print(feature_names.tolist())

print(f"\nTotal features: {len(feature_names)}")
print(f"  - Numeric (winsorized + scaled): {len(numeric_features)}")
print(f"  - Categorical (one-hot encoded): {len(feature_names) - len(numeric_features)}")

### Complete Pipeline: Preprocessing + Model

In [ ]:
# Full pipeline: preprocessing + model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

# Cross-validate
scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')
mse_scores = -scores

print("5-Fold Cross-Validation Results:")
print(f"  MSE: {mse_scores.mean():.6f} (+/- {mse_scores.std()*2:.6f})")
print(f"  Individual folds: {mse_scores.round(6).tolist()}")

---

## Summary: Key Takeaways

### Scaling
```python
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

# StandardScaler: mean=0, std=1 (default choice)
# RobustScaler: uses median/IQR (when you have outliers)
# MinMaxScaler: scales to [0, 1] (when you need bounded output)
```

### Categorical Encoding
```python
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# OrdinalEncoder: categories with natural order (credit ratings)
# OneHotEncoder: categories without order (sector, country)
# Use drop='first' for regression to avoid multicollinearity
```

### Missing Data
```python
from sklearn.impute import SimpleImputer, KNNImputer

# SimpleImputer: fill with mean/median/most_frequent
# KNNImputer: use similar observations
# add_indicator=True: add binary flags for missingness
```

### Time Series Features
```python
# Lag features (use shift with positive values only!)
df['return_lag1'] = df['return'].shift(1)

# Rolling statistics
df['volatility_20d'] = df['return'].rolling(20).std()

# Momentum
df['momentum_20d'] = df['price'].pct_change(20)
```

### Custom Transformers
```python
from sklearn.base import BaseEstimator, TransformerMixin

class MyTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Learn from data
        return self
    
    def transform(self, X):
        # Apply transformation
        return X_transformed
```

### ColumnTransformer
```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])
```

---

## What's Next?

In the final notebook, we'll cover **Model Selection, Evaluation, and Cross-Validation**:
- How to choose the right model
- Evaluation metrics for regression and classification
- Why train/test split isn't enough
- **TimeSeriesSplit**: The right way for financial data
- Hyperparameter tuning with GridSearchCV

Take a break, then see you in `3_models_and_evaluation.ipynb`!